# 06, EPSS on day 30

The exploitation model scores a CVE on the day it is published. EPSS can be read on any day after, and by day 30 it has seen a month of exploitation activity, scanning and write ups that the model never saw. Ranking development CVEs by EPSS at day 30 against the model at day 0 is the comparison a hurried report would make; this notebook makes it, on development, and shows where its advantage comes from.

In [1]:
import json

import numpy as np
import pandas as pd
import polars as pl

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(110)
SEED = 20261114  # evaluation/phase2_plan.yaml, defaults.seed
print("numpy", np.__version__, "pandas", pd.__version__, "polars", pl.__version__)

numpy 2.5.4 pandas 2.3.3 polars 2.1.0


In [2]:
from unscheduled_models import rank

labels = pl.read_parquet("data/sets/development_labels.parquet")
inputs = pl.read_parquet("data/sets/development_inputs.parquet").select("cve_id", "published")
population = rank.population(labels.join(inputs, on="cve_id"))
scores = population.join(pl.read_parquet("data/models/rank/development_scores.parquet"), on="cve_id", how="left")
offsets = pl.read_parquet("data/bronze/epss_offsets/**/*.parquet", hive_partitioning=False)
truth = rank.truth(scores)
print(f"{scores.height:,} development CVEs in the rank population, {int(truth.sum())} listed in KEV within 180 days")
by = {"model at day 0": scores["exploit_lgbm"].to_numpy()}
for day in (1, 7, 30):
    one = offsets.filter(pl.col("offset_days") == day).select("cve_id", "epss").unique("cve_id")
    by[f"EPSS at day {day}"] = scores.select("cve_id").join(one, on="cve_id", how="left")["epss"].fill_null(-1.0).to_numpy()
table = pd.DataFrame({name: [rank.recall_at_top(truth, s)] for name, s in by.items()}, index=["recall in the top 5 percent"])
(table.T * 100).round(1)

39,919 development CVEs in the rank population, 127 listed in KEV within 180 days


                recall in the top 5 percent
model at day 0                         27.6
EPSS at day 1                          33.1
EPSS at day 7                          52.8
EPSS at day 30                         70.9

In [3]:
days = scores.filter(pl.col("kev_180"))["days_to_kev"].to_numpy()
print("listed CVEs listed within 30 days of publication:", int((days <= 30).sum()), "of", len(days))
print("median days to listing:", float(np.median(days)))

listed CVEs listed within 30 days of publication: 90 of 127
median days to listing: 5.0


On development, 90 of the 127 listings came within 30 days of publication and the median came after five days, so EPSS read on day 30, and on day 7 for most of them, was read after the listing it is scored against. Its lead over the model there is the label arriving early, not a better ranking on the day of publication. EPSS at day 1 is the nearest to a fair comparison, and on development it is ahead of the model too. The registered C2 reports EPSS at days 1, 7 and 30 beside the model, calls a difference whose interval holds zero a tie, and says which days have seen more than the model; it never ranks the model against day 30 as if the two had seen the same days.